<a href="https://colab.research.google.com/github/Cleiton-sousa97/miniguia-ia-generativa-trabalho/blob/main/Agente_Financeiro_Digital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Gravação de Áudio (Python + JavaScript)**

In [14]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

**2. Reconhecimento de Fala com Whisper**

In [4]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.3 MB/s eta 0:00:00


In [5]:
import whisper

language = "pt"
model = whisper.load_model("small")

result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print("Pergunta do usuário:", transcription)


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 93.2MiB/s]


Pergunta do usuário:  Qual as tipos de investimento?


**3. Integração com ChatGPT (Agente Financeiro)**

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Lê o token salvo no Secret do Colab
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# Chamada ao ChatGPT
response = client.chat.completions.create(
    model="gpt-3.5-turbo",   # alterado para gpt-3.5-turbo
    messages=[
        {"role": "system", "content": "Você é um agente financeiro que explica investimentos e produtos bancários de forma clara e acessível."},
        {"role": "user", "content": transcription}
    ]
)

chatgpt_response = response.choices[0].message.content
print("Resposta do Agente Financeiro:", chatgpt_response)


**4. Conversão em Voz com gTTS**

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))
